# Method Comparison Notebook

Compare UAV localization methods on the same dataset split and summarize the metrics used in Table 5 of `paper_draft/main_0406.tex`:

- Mean (m)
- Med. (m)
- RMSE (m)
- R@10m (%)
- R@50m (%)
- Time (s)

Initial supported methods:

- `main` — LocalizationUAV MFCA pipeline
- `sift` — standalone classical baseline
- `orb` — standalone classical baseline


In [ ]:
import os
import sys
import time
import math
from pathlib import Path
import pathlib
pathlib.PosixPath = pathlib.WindowsPath

import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
PAPER6_ROOT = REPO_ROOT.parent
if str(PAPER6_ROOT) not in sys.path:
    sys.path.insert(0, str(PAPER6_ROOT))

DATA_ROOT = Path(r'D:/bk_study_stuff/paper6/UAV-VisLoc')
MODEL_PATH = REPO_ROOT / 'best_model.pth'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'method_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ['main', 'sift', 'orb']
SITES = ['01','02','03','04','05','06','07','08','10','11']

# Option 1: lấy tối đa N ảnh đầu tiên cho mỗi site. Đặt None để lấy toàn bộ.
MAX_IMAGES_PER_SITE = 3

# Option 2: chọn ảnh cụ thể theo từng site. Nếu site có trong dict này thì ưu tiên list này thay cho MAX_IMAGES_PER_SITE.
# Đặt SELECTED_IMAGES = {} hoặc None nếu muốn dùng MAX_IMAGES_PER_SITE cho mọi site.
SELECTED_IMAGES = {
    '01': ['01_0515.JPG',],
    '02': ['02_0026.JPG'],
    '03': ['03_0001.JPG'],
    '04': ['04_0080.JPG'],
    '05': ['05_0052.JPG'],
    '06': ['06_0136.JPG'],
    '07': ['07_0010.JPG'],
    '08': ['08_0311.JPG'],
    '10': ['10_0065.JPG'],
    '11': ['11_0114.JPG'], 

}

TOP_K = 5
MAIN_TOP_N = 100
MAIN_SCORE_THRESHOLD = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Repo root :', REPO_ROOT)
print('Data root :', DATA_ROOT)
print('Model     :', MODEL_PATH)
print('Output dir:', OUTPUT_DIR)
print('Device    :', DEVICE)

Repo root : D:\bk_study_stuff\paper6\LocalizationUAV
Data root : D:\bk_study_stuff\paper6\UAV-VisLoc
Model     : D:\bk_study_stuff\paper6\LocalizationUAV\best_model.pth
Output dir: D:\bk_study_stuff\paper6\LocalizationUAV\outputs\method_comparison
Device    : cuda


In [17]:
from localization import load_model, process_uav, SatelliteDatabase, query_uav
from localization.database.builder import extract_patch_descriptors
from localization.io.bounds import load_satellite_bounds, pixel_to_latlon
from localization.io.dataset import VisLocFlight, load_flight_metadata

import eval_sift_orb_v2 as classical_baselines


In [18]:
def build_eval_rows_dataframe(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=[
            'method', 'site', 'filename', 'gt_lat', 'gt_lon',
            'pred_lat', 'pred_lon', 'error_m', 'matched', 'time_s'
        ])
    return df


def summarize_metrics(results_df):
    summaries = []
    for method, group in results_df.groupby('method', sort=False):
        matched = group[group['matched'] == True].copy()
        errors = matched['error_m'].astype(float).to_numpy() if not matched.empty else np.array([], dtype=float)
        times = group['time_s'].astype(float).to_numpy() if 'time_s' in group else np.array([], dtype=float)

        summaries.append({
            'Method': method,
            'Mean (m)': float(np.mean(errors)) if len(errors) else np.nan,
            'Med. (m)': float(np.median(errors)) if len(errors) else np.nan,
            'RMSE (m)': float(np.sqrt(np.mean(errors ** 2))) if len(errors) else np.nan,
            'R@10m (%)': float(np.mean(errors <= 10) * 100.0) if len(errors) else np.nan,
            'R@50m (%)': float(np.mean(errors <= 50) * 100.0) if len(errors) else np.nan,
            'Time (s)': float(np.mean(times)) if len(times) else np.nan,
            'Matched / Total': f"{int(group['matched'].sum())}/{len(group)}",
        })
    summary_df = pd.DataFrame(summaries)
    if not summary_df.empty:
        summary_df = summary_df[['Method', 'Mean (m)', 'Med. (m)', 'RMSE (m)', 'R@10m (%)', 'R@50m (%)', 'Time (s)', 'Matched / Total']]
    return summary_df


def select_flight_images(flight, metadata_df, image_names=None, max_images_per_site=None):
    rows = metadata_df.copy()
    rows['filename'] = rows['filename'].astype(str).str.strip()
    rows['lat_num'] = pd.to_numeric(rows['lat'], errors='coerce')
    rows['lon_num'] = pd.to_numeric(rows['lon'], errors='coerce')
    rows = rows[rows['lat_num'].notna() & rows['lon_num'].notna()].copy()
    rows = rows.drop(columns=['lat_num', 'lon_num'])
    if image_names is not None:
        wanted = set(image_names)
        rows = rows[rows['filename'].isin(wanted)]
    if max_images_per_site is not None:
        rows = rows.head(int(max_images_per_site))
    selected = []
    for _, row in rows.iterrows():
        image_path = flight.drone_image_path(str(row['filename']).strip())
        if image_path.exists():
            selected.append((image_path, row))
    return selected

In [19]:
import gc

_main_model = None


def get_main_model():
    global _main_model
    if _main_model is None:
        _main_model = load_model(
            model_path=str(MODEL_PATH),
            device=DEVICE,
            num_classes=2,
            pretrained=False,
        ).to(DEVICE).eval()
    return _main_model


def load_main_database(site):
    db_path = REPO_ROOT / 'outputs' / site / f'satellite{site}_kdtree.npz'
    if not db_path.exists():
        raise FileNotFoundError(
            f'Missing KD-tree database for site {site}: {db_path}. Build it first with notebook 01.'
        )
    return SatelliteDatabase.load(str(db_path))


def select_flight_images(flight, metadata_df, image_names=None, max_images_per_site=None):
    rows = metadata_df.copy()
    rows['filename'] = rows['filename'].astype(str).str.strip()
    rows['lat_num'] = pd.to_numeric(rows['lat'], errors='coerce')
    rows['lon_num'] = pd.to_numeric(rows['lon'], errors='coerce')
    rows = rows[rows['lat_num'].notna() & rows['lon_num'].notna()].copy()
    rows = rows.drop(columns=['lat_num', 'lon_num'])
    if image_names is not None:
        wanted = set(image_names)
        rows = rows[rows['filename'].isin(wanted)]
    if max_images_per_site is not None:
        rows = rows.head(int(max_images_per_site))
    selected = []
    for _, row in rows.iterrows():
        image_path = flight.drone_image_path(str(row['filename']).strip())
        if image_path.exists():
            selected.append((image_path, row))
    return selected


def run_main_method_for_site(site, image_names=None, max_images_per_site=None):
    model = get_main_model()
    db = load_main_database(site)
    flight = VisLocFlight(flight_id=site, root=DATA_ROOT)
    metadata_df = load_flight_metadata(flight.metadata_csv)
    selected = select_flight_images(flight, metadata_df, image_names=image_names, max_images_per_site=max_images_per_site)

    bounds = load_satellite_bounds(
        satellite_filename=os.path.basename(str(flight.satellite_tif)),
        csv_path=str(flight.bounds_csv),
    )
    with Image.open(flight.satellite_tif) as sat:
        sat_w, sat_h = sat.size

    rows = []
    for image_path, row in selected:
        gt_lat = float(row['lat'])
        gt_lon = float(row['lon'])
        t0 = time.time()
        try:
            _, img_uav_500, _meta = process_uav(
                img_path=str(image_path),
                csv_path=str(flight.metadata_csv),
            )
            uav_descriptors, _ = extract_patch_descriptors(
                patch_image=img_uav_500,
                model=model,
                device=DEVICE,
                score_threshold=MAIN_SCORE_THRESHOLD,
            )
            result = query_uav(uav_descriptors, db, k=TOP_K, top_n=MAIN_TOP_N) if uav_descriptors.shape[0] > 0 else None
            elapsed = time.time() - t0

            if result is None or bounds is None:
                pred_lat = None
                pred_lon = None
                error_m = float('inf')
                matched = False
                pred_x = None
                pred_y = None
                vote_count = 0
            else:
                pred_x, pred_y = float(result.pixel_xy[0]), float(result.pixel_xy[1])
                pred_lat, pred_lon = pixel_to_latlon(pred_x, pred_y, bounds, sat_w, sat_h)
                error_m = float(classical_baselines.haversine_m(gt_lat, gt_lon, pred_lat, pred_lon))
                matched = True
                vote_count = int(result.vote_count)

            rows.append({
                'method': 'main',
                'site': site,
                'filename': image_path.name,
                'gt_lat': gt_lat,
                'gt_lon': gt_lon,
                'pred_lat': pred_lat,
                'pred_lon': pred_lon,
                'pred_x': pred_x,
                'pred_y': pred_y,
                'vote_count': vote_count,
                'error_m': error_m,
                'matched': matched,
                'time_s': elapsed,
            })
        except Exception as exc:
            rows.append({
                'method': 'main',
                'site': site,
                'filename': image_path.name,
                'gt_lat': gt_lat,
                'gt_lon': gt_lon,
                'pred_lat': None,
                'pred_lon': None,
                'pred_x': None,
                'pred_y': None,
                'vote_count': 0,
                'error_m': float('inf'),
                'matched': False,
                'time_s': time.time() - t0,
                'error_note': str(exc),
            })
    del db
    gc.collect()
    return rows


def run_classical_method_for_site(site, method, image_names=None, max_images_per_site=None):
    flight = VisLocFlight(flight_id=site, root=DATA_ROOT)
    metadata_df = load_flight_metadata(flight.metadata_csv)
    selected = select_flight_images(flight, metadata_df, image_names=image_names, max_images_per_site=max_images_per_site)

    normalized = []
    for image_path, _row in selected:
        results = classical_baselines.evaluate(
            data_root=str(DATA_ROOT),
            site=site,
            method=method,
            img_name=image_path.name,
            output_csv=None,
        )
        for r in results:
            normalized.append({
                'method': r['method'],
                'site': r['site'],
                'filename': r['filename'],
                'gt_lat': r['gt_lat'],
                'gt_lon': r['gt_lon'],
                'pred_lat': r['pred_lat'],
                'pred_lon': r['pred_lon'],
                'pred_x': r.get('pred_x'),
                'pred_y': r.get('pred_y'),
                'inliers': r.get('inliers', 0),
                'error_m': r['error_m'],
                'matched': r['matched'],
                'time_s': r['time_s'],
            })
    return normalized

In [20]:
METHOD_RUNNERS = {
    'main': run_main_method_for_site,
    'sift': lambda site, image_names=None, max_images_per_site=None: run_classical_method_for_site(site, 'sift', image_names=image_names, max_images_per_site=max_images_per_site),
    'orb': lambda site, image_names=None, max_images_per_site=None: run_classical_method_for_site(site, 'orb', image_names=image_names, max_images_per_site=max_images_per_site),
}


def resolve_site_selection(site):
    selected_map = SELECTED_IMAGES or {}
    if site in selected_map and selected_map[site]:
        return list(selected_map[site]), None
    return None, MAX_IMAGES_PER_SITE


all_rows = []
for method in METHODS:
    for site in SITES:
        image_names, max_imgs = resolve_site_selection(site)
        sel_desc = image_names if image_names is not None else f'first {max_imgs}'
        print(f'Running {method} on site {site} (images: {sel_desc}) ...')
        rows = METHOD_RUNNERS[method](site, image_names=image_names, max_images_per_site=max_imgs)
        all_rows.extend(rows)

results_df = build_eval_rows_dataframe(all_rows)
results_df

Running main on site 01 (images: ['01_0515.JPG']) ...
Running main on site 02 (images: ['02_0026.JPG']) ...
Running main on site 03 (images: ['03_0001.JPG']) ...
Running main on site 04 (images: ['04_0080.JPG']) ...
Running main on site 05 (images: ['05_0052.JPG']) ...
Running main on site 06 (images: ['06_0136.JPG']) ...
Running main on site 07 (images: ['07_0010.JPG']) ...
Running main on site 08 (images: ['08_0311.JPG']) ...
Running main on site 10 (images: ['10_0065.JPG']) ...
Running main on site 11 (images: ['11_0114.JPG']) ...
Running sift on site 01 (images: ['01_0515.JPG']) ...
Bounds: {'LT_lat': 29.774065, 'LT_lon': 115.970635, 'RB_lat': 29.702283, 'RB_lon': 115.996851}
[SIFT] Reference status for site 01: memory
  Loaded reference cache: memory
  Offline time: 0.0s

[SIFT] Online phase: evaluating 1 UAV queries ...
  [1/1] 01_0515.JPG → 1458.0 m, px=(81.6,942.9) (inliers=36, t=2.0s)

──────────────────────────────────────────────────
[SIFT] Site 01 — 1/1 matched
  Mean   : 1

,method,site,filename,gt_lat,gt_lon,pred_lat,pred_lon,pred_x,pred_y,vote_count,error_m,matched,time_s,inliers
0,main,01,01_0515.JPG,29.745866,115.987174,29.766235,115.992630,8199.630859,2919.150146,3.0,2328.002685,True,3.899512,NaN
1,main,02,02_0026.JPG,29.781758,116.040684,29.795056,116.038626,1810.626099,8321.229492,2.0,1493.717238,True,2.822907,NaN
2,main,03,03_0001.JPG,32.304627,119.896885,32.349969,119.886543,30054.660156,2058.599609,3.0,5140.372884,True,6.570413,NaN
3,main,04,04_0080.JPG,32.233368,119.912332,32.169819,119.921849,5916.201172,31397.773438,3.0,7130.860923,True,4.334719,NaN
4,main,05,05_0052.JPG,24.664624,102.354911,24.666261,102.356652,6187.053223,237.973862,2.0,253.399791,True,1.478996,NaN
5,main,06,06_0136.JPG,32.369984,109.649305,32.359599,109.648336,4912.051270,5061.708496,3.0,1159.690681,True,2.312396,NaN
6,main,08,08_0311.JPG,30.925506,120.242344,30.926264,120.193871,21393.570312,7814.868164,8.0,4629.581012,True,2.931927,NaN
7,main,10,10_0065.JPG,40.348679,115.783781,40.349231,115.793514,6395.586426,2184.843750,3.0,828.013735,True,1.382103,NaN
8,main,11,11_0114.JPG,38.843456,101.036764,38.848144,101.031615,6899.222168,1549.872559,2.0,686.725925,True,2.359113,NaN
9,sift,01,01_0515.JPG,29.745866,115.987174,29.740206,115.973570,81.622332,942.912963,NaN,1458.029974,True,1.956375,36.0


In [21]:
summary_df = summarize_metrics(results_df)
summary_df

,Method,Mean (m),Med. (m),RMSE (m),R@10m (%),R@50m (%),Time (s),Matched / Total
0,main,2627.818319,1493.717238,3478.868144,0.0,0.0,3.121343,9/9
1,sift,3214.980633,1458.029974,4181.471475,0.0,0.0,2.727997,9/9
2,orb,1006.517396,777.876905,1067.374449,0.0,0.0,0.854473,3/9


In [22]:
raw_csv_path = OUTPUT_DIR / 'method_comparison_raw_results.csv'
summary_csv_path = OUTPUT_DIR / 'method_comparison_summary.csv'
results_df.to_csv(raw_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)
print('Saved raw results   ->', raw_csv_path)
print('Saved summary table ->', summary_csv_path)

Saved raw results   -> D:\bk_study_stuff\paper6\LocalizationUAV\outputs\method_comparison\method_comparison_raw_results.csv
Saved summary table -> D:\bk_study_stuff\paper6\LocalizationUAV\outputs\method_comparison\method_comparison_summary.csv
